In [2]:
# Пример структуры osconfeed.json
{ "Schedule":
  { "conferences": [{"serial": 115 }],
    "events": [
      { "serial": 34505,
        "name": "Why Schools Don/t Use Open Source to Teach Programming",
        "event_type": "40-minute conference session",
        "time_start": "2014-07-23 11:30:00",
        "time_stop": "2014-07-23 12:10:00",
        "venue_serial": 1462,
        "description": "Aside from the fact that high school programming...",
        "website_url": "http://oscon.com/oscon2014/public/schedule/detail/34505",
        "speakers": [157509],
        "categories": ["Education"] }
    ],
    "speakers": [
      { "serial": 157509,
        "name": "Robert Lefkowitz",
        "photo": "null",
        "url": "http://sharewave.com/",
        "position": "CTO",
        "affiliation": "Sharewave",
        "twitter": "sharewaveteam",
        "bio": "Robert /r0ml/ Lefkowitz is the CTO at Sharewave, a startup..." }
    ],
    "venues": [
      { "serial": 1462,
        "name": "F151",
        "category": "Conference Venues" }
    ]
  }
}

{'Schedule': {'conferences': [{'serial': 115}],
  'events': [{'serial': 34505,
    'name': 'Why Schools Don/t Use Open Source to Teach Programming',
    'event_type': '40-minute conference session',
    'time_start': '2014-07-23 11:30:00',
    'time_stop': '2014-07-23 12:10:00',
    'venue_serial': 1462,
    'description': 'Aside from the fact that high school programming...',
    'website_url': 'http://oscon.com/oscon2014/public/schedule/detail/34505',
    'speakers': [157509],
    'categories': ['Education']}],
  'speakers': [{'serial': 157509,
    'name': 'Robert Lefkowitz',
    'photo': 'null',
    'url': 'http://sharewave.com/',
    'position': 'CTO',
    'affiliation': 'Sharewave',
    'twitter': 'sharewaveteam',
    'bio': 'Robert /r0ml/ Lefkowitz is the CTO at Sharewave, a startup...'}],
  'venues': [{'serial': 1462,
    'name': 'F151',
    'category': 'Conference Venues'}]}}

In [5]:
# Стандартная загрузка json через контекстный менеджер
import json
with open('C:/Users/Fedor/Downloads/osconfeed.json') as fp:
    feed = json.load(fp)

In [ ]:
# Вывести все ключи, которые находятся внутри ключа (корная) 'Schedule'
sorted(feed['Schedule'].keys()) 

['conferences', 'events', 'speakers', 'venues']

In [8]:
# Вывести счетчики записей в каждой коллекции.
for key, value in sorted(feed['Schedule'].items()):
    print(f'{len(value):3} {key}')

  1 conferences
484 events
357 speakers
 53 venues


In [10]:
# Пройтись по вложенной структуре и получить последний элемент на нужном уровне
feed['Schedule']['speakers'][-1]['name']

'Carina C. Zona'

In [1]:
from collections import abc
import keyword

class FrozenJSON:
    """Допускающий только чтение фасад для навигации по JSON-подобному
       объекту с применением нотации атрибутов
    """
    def __init__(self, mapping):
        self.__data = {}
        for key, value in mapping.items():
            if keyword.iskeyword(key): # для обхода зарезервированных слов в Python
                key += '_'
            self.__data[key] = value

    def __getattr__(self, name):
        try:
            return getattr(self.__data, name)
        except AttributeError:
            return FrozenJSON.build(self.__data[name]) # В противном случае получаем элемент с ключом name из self.__data и возвращаем результат вызова для него метода FrozenJSON.build()
    
    def __dir__(self):
        # по умолчаиню возвращает список атрибутов и методов объекта
        # в данном случае возвращает ключи
        return self.__data.keys()
    
    @classmethod
    def build(cls, obj):
        if isinstance(obj, abc.Mapping): # Если obj – отображение, строим по нему объект FrozenJSON
            return cls(obj)
        elif isinstance(obj, abc.MutableSequence): # Если  это  экземпляр  MutableSequence, то  он должен  быть  списком,  поэтому строим список, рекурсивно передавая каждый элемент obj методу .build()
            return [cls.build(item) for item in obj]
        else:
            return obj # Если это не dict и не list, возвращаем элемент без изменения.

In [2]:
from collections import abc
import keyword
class FrozenJSON2:
    """Допускающий только чтение фасад для навигации по JSON-подобному
       объекту с применением нотации атрибутов
    """
    def __new__(cls, arg):
        # Будучи методом класса, __new__ получает в качестве первого аргумента сам класс, а остальные аргументы – те же, что получает __init__, за исключением self.
        if isinstance(arg, abc.Mapping):
            return super().__new__(cls) # По умолчанию работа делегируется методу __new__  суперкласса. В данном случае мы вызываем метод __new__ из базового класса object, передавая ему FrozenJSON в качестве единственного аргумента
        elif isinstance(arg, abc.MutableSequence):
            return [cls(item) for item in arg] # Оставшаяся часть __new__ ничем не отличается от прежнего метода build.
        else:
            return arg
        
    def __init__(self, mapping):
        self.__data = {}
        for key, value in mapping.items():
            if keyword.iskeyword(key):
                key += '_'
            self.__data[key] = value

    def __getattr__(self, name):
        try:
            return getattr(self.__data, name)
        except AttributeError:
            return FrozenJSON2(self.__data[name]) # Здесь раньше вызывался метод FrozenJSON.build, а теперь мы просто вызываем класс FrozenJSON, а Python обрабатывает это как вызов FrozenJSON.__new__
        
    def __dir__(self):
        return self.__data.keys()

In [ ]:
import json
JSON_PATH = 'C:/Users/Fedor/Downloads/osconfeed.json'
class Record22:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs) # Стандартная идиома для построения экземпляра, атрибуты которого создаются из именованных аргументов

    def __repr__(self):
        return f'<{self.__class__.__name__} serial={self.serial!r}>' # Использовать поле serial, чтобы построить представление Record2
    
    @classmethod
    def load(cls, path=JSON_PATH):
        records = {} # Метод load в конечном итоге вернет словарь экземпляров Record2.
        with open(path) as fp:
            raw_data = json.load(fp) # Разобрать JSON и вернуть объекты Python: списки, словари, числа и т. д
        for collection, raw_records in raw_data['Schedule'].items(): # Обойти все четыре списка верхнего уровня: 'conferences', 'events', 'speakers' и 'venues'
            record_type = collection[:-1] # record_type – имя списка без последнего символа, т. е. speakers  становится speaker
            for raw_record in raw_records:
                key = f'{record_type}.{raw_record["serial"]}' # Построить ключ в формате 'speaker.3471'.
                records[key] = Record2(**raw_record) # Создать экземпляр Record2 и сохранить его в словаре records под ключом key.
        return records

In [ ]:
records = Record2.load(JSON_PATH)
speaker = records['speaker.3471']
speaker

<Record serial=3471>

In [11]:
speaker.name, speaker.twitter

('Anna Martelli Ravenscroft', 'annaraven')

In [ ]:
import inspect
import json

JSON_PATH = 'C:/Users/Fedor/Downloads/osconfeed.json'

class Record2:
    __index = None # В закрытом атрибуте класса __index будет храниться ссылка на dict, возвращенный методом load

    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

    def __repr__(self):
        return f'<{self.__class__.__name__} serial={self.serial!r}>'
    
    
    @staticmethod 
    def fetch(key):
        # fetch сделан статическим методом, чтобы было понятно, что его действие не зависит от экземпляра или класса, от имени которого он вызывается
        # Метод  fetch  всегда  применяется  к  атрибуту  класса Record2.__index
        if Record2.__index is None:
            Record2.__index = load() # Заполнить Record2.__index, если необходимо.
        return Record2.__index[key] # Нужно, чтобы извлечь запись с заданным ключом key
    
    def load(path=JSON_PATH):
        records = {}
        with open(path) as fp:
            raw_data = json.load(fp)
        for collection, raw_records in raw_data['Schedule'].items():
            record_type = collection[:-1]
            cls_name = record_type.capitalize() # Преобразовать первую букву record_type в верхний регистр, чтобы получить потенциальное имя класса (например, 'event' превращается в 'Event')
            cls = globals().get(cls_name, Record2) # Получить объект с таким именем из глобальной области видимости модуля; если такого объекта нет, получаем Record2.
            if inspect.isclass(cls) and issubclass(cls, Record2):
                factory = cls # Если только что полученный объект – класс, который является подклассом Record2, то
                # связать с ним имя factory. Это означает, что factory может быть произвольным подклассом Record2, определяемым переменной record_type
            else:
                factory = Record2 # В противном случае связать имя factory с Record2
            for raw_record in raw_records:
                key = f'{record_type}.{raw_record["serial"]}'
                records[key] = factory(**raw_record)  # объект, сохраняемый в records, конструируется функцией factory, которая может быть конструктором класса Record2 или его подкласса – в зависимости от значения record_type
        return records
    
    class Event(Record2):

        def __repr__(self): # Если в экземпляре есть атрибут name, включаем его в строковое представление. В противном случае делегируем методу __repr__, унаследованному от Record2.
            try:
                return f'<{self.__class__.__name__} {self.name!r}>'
            except AttributeError:
                return super().__repr__() 
            
        @property
        def speakers(self):
            spkr_serials = self.__dict__['speakers']
            fetch = self.__class__.fetch
            return [fetch(f'speaker.{key}')
                    for key in spkr_serials]    
            
        @property
        def venue(self): # Свойство venue строит ключ key по атрибуту venue_serial и передает его методу класса fetch, унаследованному от Record2
            key = f'venue.{self.venue_serial}'
            return self.__class__.fetch(key)

In [8]:
# Свойства всегда являются атрибутами класса, но на самом деле они управляют доступом к атрибутам в экземплярах этого класса.

In [2]:
class LineItem:
    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price
    
    @property # декоратором @property обозначается метод чтения свойства
    def weight(self): # Имена всех методов, реализующих свойство, совпадают с именем открытого атрибута: weight.
        return self.__weight # Фактическое значение хранится в закрытом атрибуте __weight.
    
    @weight.setter # У декорированного метода чтения свойства имеется атрибут .setter, который является также и декоратором; тем самым методы чтения и установки связываются между собой
    def weight(self, value):
        if value > 0: # Если значение больше нуля, присваиваем его закрытому атрибуту __weight
            self.__weight = value
        else:
            raise ValueError('value must be > 0') # В противном случае возбуждаем исключение ValueError

In [7]:
class LineItem2:
    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price
    
    def get_weight(self):
        return self.__weight
    
    def set_weight(self, value):
        if value > 0:
            self.__weight = value
        else:
            raise ValueError('value must be > 0')
    weight = property(get_weight, set_weight) # Подкапотная и альтернативная работа property

In [2]:
class Class: 
    data = 'the class data attr'
     
    @property
    def prop(self):
         return 'the prop value'

In [4]:
c=Class()
vars(c) # __dict_ экземпляра c пустой

{}

In [5]:
c.data

'the class data attr'

In [15]:
c.data='new' # обращение к атрибуту перезатирает изначальное изначениеи создает запись в __dict__
c.data

'new'

In [8]:
Class.data # при этом атрибут самого класса не изменился

'the class data attr'

In [12]:
vars(c)

{'data': 'new'}

In [10]:
c.prop='foo'
c.prop

AttributeError: property 'prop' of 'Class' object has no setter

In [16]:
c.__dict__['prop']='foo' # сделать запись напрямую в dict возможно
c.__dict__

{'data': 'new', 'prop': 'foo'}

In [18]:
c.prop # при этом из-за property изменение не произойдет

'the prop value'

In [20]:
Class.prop='baz' # Но если изменить значение в классе, то свойство property будет уничтожено и значение для атрибута возьмется из __dict__ 
c.prop

'foo'

In [24]:
Class.data = property(lambda self: 'the "data" prop value')
c.data # Это может отработать и в обратную сторону, при изменениии свойства в классе

'the "data" prop value'

In [25]:
# В  этом  разделе  мы  прежде  всего  хотели  показать,  что  при  вычислении выражения вида obj.data поиск data начинается не с obj. 
# На самом деле поиск начинается с obj.__class__, и только если в классе не существует свойства с именем data, 
# то Python заглядывает в сам объект obj. Это правило применимо к переопределяющим дескрипторам вообще, а свойства являются лишь их частным случаем

In [28]:
def quantity(storage_name): # Аргумент storage_name определяет, где хранятся данные свойства; в случае свойства weight данные будут храниться в атрибуте с именем 'weight'

    def qty_getter(instance): 
        # Называть первый аргумент метода qty_getter  именем self было бы не со-
        # всем правильно, т. к. это не тело класса; instance  ссылается на экземпляр 
        # LineItem, в котором будет храниться атрибут
        return instance.__dict__[storage_name]
        # Метод qty_getter ссылается на storage_name, поэтому будет 
        # сохранен в замыкании этой функции; значение берется непосредственно из instance.__dict__, 
        # чтобы обойти свойство и избежать бесконечной рекурсии.
    
    def qty_setter(instance, value): # В определении метода qty_setter первым аргументом также является instance.
        if value > 0:
            instance.__dict__[storage_name] = value # Значение  сохраняется  непосредственно  в  instance.__dict__,  снова  в  обход свойства
        else:
            raise ValueError('value must be > 0')
    return property(qty_getter, qty_setter) # Сконструировать и вернуть объект свойства

class LineItem3:
    weight = quantity('weight')
    price = quantity('price') 

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight 
        self.price = price

    def subtotal(self):
        return self.weight * self.price

In [30]:
# Функции qty_getter и qty_setter обертываются объектом property в последней строке фабричной функции. 
# Когда впоследствии любая из этих функций будет вызвана для выполнения своих обязанностей, 
# она прочитает storage_name из своего замыкания и определит, откуда читать или куда записывать значение управляемого атрибута.

In [29]:
l2=LineItem3('fed',-10, 10)
l2

ValueError: value must be > 0

In [2]:
# Встроенные функциии для работы с атрибутами

In [1]:
# dir(obj)

# Перечисляет большую часть атрибутов объекта. В официальной документации  (https://docs.python.org/3/library/functions.html#dir)  сказано,  
# что  функция dir  предназначена для интерактивного использования, поэтому она выводит не полный список атрибутов, а только самые «интересные». dir 
# умеет инспектировать объекты с атрибутом __dict__ и без него. Сам атрибут  __dict__  не  входит  в  список,  формируемый  функцией  dir,  
# но  ключи, хранящие ся в __dict__, входят. Есть еще несколько специальных атрибутов классов, в частности __mro__, __bases__ и __name__, которые dir  не выводит. 
# Результат, печатаемый методом dir, можно модифицировать, реализовав специальный метод __dir__
# Если факультативный аргумент object не задан, то dir выводит имена в текущей области видимости

In [3]:
# getattr(object, name[, default])

# Получает атрибут, идентифицируемый строкой name, объекта object. Приме-
# няется прежде всего для получения атрибутов (или методов), чьи имена заранее неизвестны. В результате может быть найден атрибут, определенный 
# в классе или суперклассе объекта. Если такого атрибута не существует, getattr 
# возбуждает исключение AttributeError либо возвращает значение default, если оно задано.

In [4]:
# hasattr(object, name)

# Возвращает True, если атрибут с указанным именем существует в объекте object 
# или может быть найден с его помощью (например, в результате наследования)

In [5]:
# setattr(object, name, value)

# Присваивает значение value  поименованному атрибуту object, если object 
# это допускает. В результате может быть создан новый атрибут или изменен существующий

In [6]:
# vars([object])

# Возвращает атрибут __dict__ объекта object; функция vars не умеет работать с классами, в которых определен атрибут __slots__  и нет атрибута __dict__ 
# (в отличие от функции dir, которая справляется с такими экземплярами). Без аргумента vars() делает то же самое, что locals(): возвращает словарь, описывающий локальную область видимости.
# Важные атрибуты и функции для работы с атрибутами


In [7]:
# __delattr__(self, name)

# Вызывается при любой попытке удалить атрибут в предложении del,  
# например del obj.attr приводит к вызову Class.__delattr__(obj, 'attr'). 
# Если attr является свойством, то его метод удаления никогда не вызывается, если в классе реализован метод __delattr__.

In [8]:
# __dir__(self)

# Вызывается при вызове dir для объекта с целью получить список атрибутов, например dir(obj)  приводит к вызову Class.__dir__(obj).

In [9]:
# __getattr__(self, name)

# Вызывается только тогда, когда попытка найти поименованный атрибут в obj, Class и суперклассах завершается неудачно. 
# Выражения obj.no_such_attr, getattr(obj,  'no_such_attr') и hasattr(obj,  'no_such_attr')  
# могут привести к вызову Class.__getattr__(obj, 'no_such_attr'), но только если атрибут с таким именем отсутствует в obj, Class и его суперклассах.

In [10]:
# __getattribute__(self, name)

# Вызывается при любой попытке получить поименованный атрибут непосредственно  из  Python-кода  (интерпретатор  иногда  обходит  этот  метод, например чтобы получить метод __repr__). 
# К вызову этого метода приводит использование нотации с точкой и встроенных функций getattr  и hasattr. 
# Метод __getattr__ всегда вызывается после __getattribute__ и только в том случае, когда __getattribute__  возбуждает исключение AttributeError. 
# Чтобы при получении атрибутов obj не возникало бесконечной рекурсии, в реализации __getattribute__ следует использовать super().__getattribute__(obj, name).

In [11]:
# __setattr__(self, name, value)

# Вызывается  при  любой  попытке  установить  поименованный  атрибут. 
# К вызову этого метода приводит использование нотации с точкой и встроенной функции setattr, 
# например и obj.attr = 42, и setattr(obj, 'attr', 42) приводят к вызову Class.__setattr__(obj, 'attr', 42).

In [13]:
# Дескрипторы – это способ повторного использования одной и той же логики доступа  в  нескольких  атрибутах

# Дескриптор – это класс, который реализует динамический протокол, 
# содержащий методы __get__, __set__ и __delete__. 
# Класс property реализует весь протокол дескриптора. 
# Как обычно, разрешается реализовывать протокол частично. На самом деле большинство дескрипторов, 
# встречающихся в реальных программах, реализуют только методы __get__ и __set__, а многие – и вовсе лишь один из них.


# Дескрипторы – уникальная черта Python, и используются они не только на уровне приложения, но и в инфраструктуре самого языка. Пользовательские функции – это дескрипторы

In [15]:
class Quantity: # Дескриптор основан на протоколе, для его реализации не требуется наследование.

    def __init__(self, storage_name):
        self.storage_name = storage_name # В каждом экземпляре Quantity  имеется атрибут storage_name: имя атрибута хранения, в котором хранится значение управляемого экземпляра.

    def __set__(self, instance, value):
        # Метод __set__ вызывается при любой попытке присвоить значение управляемому  атрибуту.  
        # В  данном  случае  self  –  экземпляр  дескриптора  (т.  е. LineItem.weight или LineItem.price), 
        # instance – управляемый экземпляр (экземпляр LineItem), а value – присваиваемое значение
        if value > 0:
            instance.__dict__[self.storage_name] = value
            # Мы должны сохранить значение атрибута непосредственно в __dict__; 
            # попытка вызвать setattr(instance,  self.storage_name)  привела бы к повторному вызову метода __set__ и, стало быть, к бесконечной рекурсии.
        else:
            msg = f'{self.storage_name} must be > 0'
            raise ValueError(msg)
        
    def __get__(self, instance, owner): # Реализовать  __get__  необходимо,  потому  что  имя  управляемого  атрибута может не совпадать с storage_name. Про аргумент owner я расскажу ниже
        return instance.__dict__[self.storage_name]
    


class LineItem4:
    weight = Quantity('weight') # Первый экземпляр дескриптора связывается с атрибутом weight
    price = Quantity('price') # Второй экземпляр дескриптора связывается с атрибутом price

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price    

In [17]:
truffle = LineItem4('White truffle', 100, 0)
truffle

ValueError: price must be > 0

In [18]:
# Реализация __set_name__

class Quantity:

    def __set_name__(self, owner, name): 
        # self – экземпляр дескриптора (неуправляемый экземпляр), 
        # owner – управляемый класс, а name – имя атрибута owner, 
        # которому был назначен этот дескриптор в теле класса owner.
        self.storage_name = name

    def __set__(self, instance, value):
        if value > 0:
            instance.__dict__[self.storage_name] = value
        else:
            msg = f'{self.storage_name} must be > 0'
            raise ValueError(msg)
        
    # __get__ не нужен
    # Реализовывать __get__ необязательно, потому что имя атрибута хранения совпадает с именем управляемого атрибута. 
    # Выражение product.price получает атрибут price непосредственно из экземпляра LineItem.

class LineItem:
    weight = Quantity() # Теперь нам не нужно передавать имя управляемого атрибута конструктору Quantity. В этом и заключалась цель данной версии
    price = Quantity()

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price

In [21]:
# Реализация работы дескриптора через шаблонный метод и наследование

import abc
class Validated(abc.ABC):

    def __set_name__(self, owner, name):
        self.storage_name = name

    def __set__(self, instance, value):
        value = self.validate(self.storage_name, value) # Метод __set__ делегирует проверку методу validate
        instance.__dict__[self.storage_name] = value # .. а затем использует возвращенное значение value, чтобы обновить хранимое значение.

    @abc.abstractmethod
    def validate(self, name, value): # Метод validate абстрактный, это шаблонный метод.
        """вернуть проверенное значение или возбудить ValueError"""

class Quantity(Validated):
    """число, большее нуля"""
    def validate(self, name, value): # Реализация  шаблонного  метода,  которой  требует  абстрактный  метод Validated.validate
        if value <= 0:
            raise ValueError(f'{name} must be > 0')
        return value
    
class NonBlank(Validated):
    """строка, содержащая хотя бы один символ, отличный от пробела"""
    def validate(self, name, value):
        value = value.strip()
        if not value: # Если после удаления начальных и конечных пробелов ничего не осталось, отвергнуть значение.
            raise ValueError(f'{name} cannot be blank')
        return value 
    # Требуя, чтобы конкретные методы validate возвращали проверенное значение, 
    # мы оставляем им возможность очистить, преобразовать или нормализовать полученные данные. 
    # В данном случае значение value перед возвратом очищается от начальных и конечных пробелов.       

In [22]:
# Если бы notebook выше был .py файлом, можно было бы импортировать его
# и использовать классы как дескрипторы

import model_v5 as model

class LineItem:
    description = model.NonBlank()
    weight = model.Quantity()
    price = model.Quantity()

    def __init__(self, description, weight, price):
        self.description = description
        self.weight = weight
        self.price = price

    def subtotal(self):
        return self.weight * self.price

ModuleNotFoundError: No module named 'model_v5'

In [1]:
def cls_name(obj_or_cls):
    cls = type(obj_or_cls)
    if cls is type:
        cls = obj_or_cls
    return cls.__name__.split('.')[-1]

def display(obj):
    cls = type(obj)
    if cls is type:
        return f'<class {obj.__name__}>'
    elif cls in [type(None), int]:
        return repr(obj)
    else:
        return f'<{cls_name(obj)} object>'

def print_args(name, *args):
    pseudo_args = ', '.join(display(x) for x in args)
    print(f'-> {cls_name(args[0])}.__{name}__({pseudo_args})')

class Overriding:
    """он же дескриптор данных или принудительный дескриптор"""
    # Типичный  переопределяющий  дескрипторный  класс  с  методами  __get__ и __set__.
    def __get__(self, instance, owner):
        print_args('get', self, instance, owner)

    def __set__(self, instance, value):
        print_args('set', self, instance, value)

class OverridingNoGet:
    """переопределяющий дескриптор без ``__get__``"""
    # Переопределяющий дескриптор без метода __get__.
    def __set__(self, instance, value):
        print_args('set', self, instance, value)

class NonOverriding:
    """он же дескриптор без данных или маскируемый дескриптор"""
    # Здесь нет метода __set__, т. е. этот дескриптор непереопределяющий.
    def __get__(self, instance, owner):
        print_args('get', self, instance, owner)

class Managed:
    # правляемый класс, в котором используется по одному экземпляру каждого дескрипторного класса. 
    over = Overriding()
    over_no_get = OverridingNoGet()
    non_over = NonOverriding()

    def spam(self): # Метод spam включен для сравнения, потому что методы – также дескрипторы.
        print(f'-> Managed.spam({display(self)})')        


In [14]:
# Пример работы переопределеяющего дескриптора с __get__, __set__

In [ ]:
obj = Managed()
obj.over # obj.over активирует метод дескриптора __get__, передавая ему управляемый экземпляр obj во втором аргументе.

-> Overriding.__get__(<Overriding object>, <Managed object>, <class Managed>)


In [4]:
Managed.over # Managed.over  активирует метод дескриптора __get__, передавая ему None  во втором аргументе (instance)

-> Overriding.__get__(<Overriding object>, None, <class Managed>)


In [6]:
obj.over=7 # Присваивание  obj.over  активирует  метод  дескриптора  __set__,  передавая ему значение 7 в последнем аргументе.

-> Overriding.__set__(<Overriding object>, <Managed object>, 7)


In [ ]:
obj.over # Чтение obj.over по-прежнему активирует метод дескриптора __get__.

-> Overriding.__get__(<Overriding object>, <Managed object>, <class Managed>)


In [8]:
obj.__dict__['over'] = 8 # Установка значения непосредственно в obj.__dict__ в обход дескриптора

In [11]:
vars(obj) # Проверить, что значение попало в obj.__dict__ и ассоциировано с ключом over.

{'over': 8}

In [13]:
obj.over # днако даже при наличии атрибута экземпляра с именем over дескриптор Managed.over все равно переопределяет попытки читать obj.over

-> Overriding.__get__(<Overriding object>, <Managed object>, <class Managed>)


In [15]:
# Переопределяющий дескриптор без __get__, только __set__

In [17]:
obj.over_no_get # В этом переопределяющем дескрипторе нет метода __get__, поэтому чтение obj.over_no_get извлекает экземпляр дескриптора из класса.

In [19]:
Managed.over_no_get # То же происходит, если извлечь экземпляр дескриптора непосредственно из управляемого класса.

In [20]:
obj.over_no_get = 7 # Попытка присвоить значение атрибуту obj.over_no_get активирует метод дескриптора __set__.

-> OverridingNoGet.__set__(<OverridingNoGet object>, <Managed object>, 7)


In [21]:
obj.over_no_get # Поскольку наш метод __set__ не производит никаких изменений, повторное чтение obj.over_no_get  извлекает все тот же экземпляр дескриптора из управляемого класса.

In [22]:
obj.__dict__['over_no_get'] = 9 # Установить атрибут экземпляра с именем over_no_get через атрибут __dict__ экземпляра.

In [24]:
obj.over_no_get # Теперь  новый  атрибут  экземпляра  over_no_get  маскирует  дескриптор,  но только при чтении.

9

In [25]:
obj.over_no_get = 7 # Попытка присвоить значение атрибуту obj.over_no_get по-прежнему проходит через метод __set__ дескриптора.

-> OverridingNoGet.__set__(<OverridingNoGet object>, <Managed object>, 7)


In [26]:
obj.over_no_get # Но при чтении дескриптор замаскирован до тех пор, пока существует одноименный атрибут экземпляра.

9

In [27]:
# Непереопределяющий дескриптор с __get__, но без __set__

In [29]:
obj.non_over # obj.non_over активирует метод дескриптора __get__, передавая ему obj во втором аргументе

-> NonOverriding.__get__(<NonOverriding object>, <Managed object>, <class Managed>)


In [30]:
obj.non_over = 7 # Managed.non_over – непереопределяющий дескриптор, поэтому не существует метода __set__, который мог бы вмешаться в эту операцию присваивания.

In [32]:
obj.non_over # Теперь в obj есть атрибут экземпляра с именем non_over, который маскирует одноименный дескрипторный атрибут в классе Managed.

7

In [33]:
Managed.non_over # Дескриптор Managed.non_over  по-прежнему существует и перехватывает эту операцию доступа через класс.

-> NonOverriding.__get__(<NonOverriding object>, None, <class Managed>)


In [34]:
del obj.non_over # Если атрибут экземпляра non_over удалить...

In [35]:
obj.non_over # ... то чтение obj.non_over активирует метод __get__ дескриптора в классе, однако вторым аргументом будет управляемый экземпляр.

-> NonOverriding.__get__(<NonOverriding object>, <Managed object>, <class Managed>)


In [36]:
# Советы по использованию дескрипторов

In [37]:
# Для простоты пользуйтесь классом property

# Встроенный класс property создает переопределяющие дескрипторы, 
# в которых реализованы оба метода __set__ и __get__, даже если вы сами не задавали метод установки. 
# Подразумеваемый по умолчанию метод __set__ возбуждает исключение AttributeError: can't set attribute, 
# поэтому свойство – это простейший способ создать доступный только для чтения атрибут

In [38]:
# В дескрипторах только для чтения необходим метод __set__

# Если  вы  используете  дескрипторный  класс  для  реализации  атрибута, допус кающего только чтение, то не забывайте реализовывать оба метода 
# __get__ и __set__, иначе одноименный атрибут экземпляра замаскирует дескриптор. 
# Метод __set__  атрибута, доступного только для чтения, должен просто возбуждать исключение AttributeError с подходящим сообщением

In [39]:
# Проверяющим дескрипторам достаточно одного метода __set__

# Если  дескриптор  предназначен только  для  проверки  значений, то  метод 
# __set__  должен  проверять  полученный  аргумент  value  и,  если  он  правилен, устанавливать значение непосредственно в атрибуте __dict__ экземпляра, 
# используя в качестве ключа имя экземпляра дескриптора. 
# Тогда чтение атрибута с таким же именем из экземпляра будет производиться максимально быстро, т. к. не требует наличия метода __get__

In [40]:
# Кеширование можно эффективно реализовать при наличии одного лишь __get__

# Если вы напишете только метод __get__, то получите непереопределяющий дескриптор. 
# Они полезны, когда требуется выполнить накладные вычисления и кешировать результат, установив атрибут экземпляра с таким же именем. 
# Одноименный атрибут экземпляра маскирует дескриптор, поэтому при последующем доступе к этому атрибуту 
# значение будет извлекаться непосредственно из атрибута __dict__ экземпляра в обход метода __get__ дескриптора. 
# Декоратор @functools.cached_property на самом деле порождает непереопределяющий дескриптор.

In [41]:
# Неспециальные методы можно замаскировать атрибутами экземпляра

# Поскольку  в  функциях  и  методах  реализован  только  метод  __get__,  они не перехватывают попытки установить одноименные атрибуты экземпляра, 
# так что после простого присваивания my_obj.the_method = 7 последующий доступ к the_method через данный экземпляр вернет число 7, 
# хотя на других экземплярах это никак не отразится. Однако на специальные методы это не  распространяется.  Интерпретатор  ищет  специальные  методы  только в самом классе, 
# т. е. repr(x) всегда вычисляется как x.__class__.__repr__(x), так что наличие атрибута __repr__ в x не влияет на результат repr(x). 
# По той же причине существование атрибута с именем __getattr__ в экземпляре не испортит обычный алгоритм доступа к атрибутам.